In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scrublet as scr
import scipy.io
import matplotlib.pyplot as plt
import os
import bbknn as bk
import scvelo as scv
import anndata
sc.settings.set_figure_params(dpi=300,fontsize=10)

In [ ]:
sc.settings.set_figure_params(dpi=300,fontsize=10)

In [ ]:
fibroblastslognorm = sc.read_h5ad('data/fibroblastslognorm.h5ad')

In [ ]:
sc.tl.pca(fibroblastslognorm, svd_solver='arpack',random_state=2)
bk.bbknn(fibroblastslognorm)
sc.tl.umap(fibroblastslognorm,random_state=2)

In [ ]:
from kneed import KneeLocator
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
kmeans_kwargs = {"init": "random","n_init": 10,"max_iter": 300,"random_state": 2}

In [ ]:
sse = []
   ...: for k in range(1, 34):
   ...:     kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
   ...:     kmeans.fit(fibroblastslognorm.obsm['X_pca'])
   ...:     sse.append(kmeans.inertia_)

In [ ]:
sc.settings.set_figure_params(dpi=300,fontsize=5)
plt.style.use("fivethirtyeight")
plt.plot(range(1, 34), sse)
plt.xticks(range(1, 34))
plt.xlabel("Number of Clusters")
plt.ylabel("SSE")
plt.show()

In [ ]:
kl = KneeLocator(range(1, 34), sse, curve="convex", direction="decreasing")
kl.elbow

In [ ]:
#silhouette score 
silhouette_coefficients = []
for k in range(2, 34):
    kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
    kmeans.fit(fibroblastslognorm.obsm['X_pca'])
    score = silhouette_score(fibroblastslognorm.obsm['X_pca'], kmeans.labels_)
    silhouette_coefficients.append(score)

In [ ]:
plt.style.use("fivethirtyeight")
plt.plot(range(2, 34), silhouette_coefficients)
plt.xticks(range(2, 34))
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Coefficient")
plt.show()

In [ ]:
CH = []
for k in range(2, 34):
    kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
    kmeans.fit(fibroblastslognorm.obsm['X_pca'])
    score = calinski_harabasz_score(fibroblastslognorm.obsm['X_pca'], kmeans.labels_)
    CH.append(score)

In [ ]:
plt.style.use("fivethirtyeight")
plt.plot(range(2, 34), CH)
plt.xticks(range(2, 34))
plt.xlabel("Number of Clusters")
plt.ylabel("CH score")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, init ='k-means++', max_iter=300, n_init=10,random_state=2 )

In [ ]:
y_kmeans = kmeans.fit_predict(fibroblastslognorm.obsm['X_pca'])
fibroblastslognorm.obs['kmeans_4'] = y_kmeans
fibroblastslognorm.obs['kmeans_4'] = fibroblastslognorm.obs['kmeans_4'].astype('category')

In [ ]:
plt.scatter(fibroblastslognorm.obsm['X_pca'][:, 0], fibroblastslognorm.obsm['X_pca'][:, 1], s=5)

In [ ]:
sc.pl.umap(fibroblastslognorm,color='kmeans_4',legend_fontsize='large',use_raw=True)

In [ ]:
from ClusterSupport import KMeans

In [ ]:
gapstat = KMeans().gap_statistic(fibroblastslognorm.obsm['X_pca'], 'n_clusters', range(2, 34))
gapstat['stat+SE'] = gapstat['gap_statistic'] + gapstat['standard_error']
gapstat['stat-SE'] = gapstat['gap_statistic'] - gapstat['standard_error']

In [ ]:
plt.errorbar(x = gapstat.index, y = gapstat['gap_statistic'], yerr = gapstat['standard_error'], linewidth=0.1,elinewidth=0.2)

In [ ]:
sc.tl.leiden(fibroblastslognorm,resolution=1.1,n_iterations=-1,random_state=2)
sc.pl.umap(fibroblastslognorm,color='leiden',legend_fontsize='large',use_raw=True)

In [ ]:
sc.tl.dendrogram(fibroblastslognorm,groupby='leiden')

In [ ]:
sc.pl.dendrogram(fibroblastslognorm,groupby='leiden')

In [ ]:
sc.settings.set_figure_params(dpi=300,fontsize=10)

In [ ]:
sc.pl.correlation_matrix(fibroblastslognorm,groupby='leiden',show_correlation_numbers=True)

In [ ]:
corrmat = fibroblastslognorm.uns['dendrogram_leiden']['correlation_matrix']
scorelist = []
for item in corrmat:
    score = sum(item)/len(item)
    scorelist.append(score)
    print(score)

In [ ]:
sc.tl.rank_genes_groups(fibroblastslognorm,groupby='leiden',n_genes=10,method='logreg',corr_method='bonferroni', use_raw = True)

In [ ]:
sc.tl.dendrogram(fibroblastslognorm,groupby='leiden')

In [ ]:
sc.pl.dendrogram(fibroblastslognorm,groupby='leiden')

In [ ]:
sc.settings.set_figure_params(dpi=300,fontsize=14)

In [ ]:
sc.tl.filter_rank_genes_groups(fibroblastslognorm, min_fold_change=1)

In [ ]:
sc.pl.rank_genes_groups_dotplot(fibroblastslognorm,n_genes=3,key='rank_genes_groups_filtered',smallest_dot=10,use_raw = True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(fibroblastslognorm,n_genes=3,smallest_dot=10,use_raw = True)

In [ ]:
sc.tl.rank_genes_groups(fibroblastslognorm, 'leiden', method = 'wilcoxon', n_genes = 100, use_raw = True)
result = fibroblastslognorm.uns['rank_genes_groups']
groups = result['names'].dtype.names
markers = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names','logfoldchange','p-value','scores']})
markers.head(10)

In [ ]:
markers.to_csv("fibro_low_dge_logreg.csv")

In [ ]:
from SCCAF import SCCAF_assessment, plot_roc
import scanpy as sc

y_prob, y_pred, y_test, clf, cvsm, acc = SCCAF_assessment(fibroblastslognorm.X, fibroblastslognorm.obs['leiden'], n=100)

In [ ]:
import matplotlib.pyplot as plt

plot_roc(y_prob, y_test, clf, cvsm=cvsm, acc=acc,plot='roc',fontsize=12)
plt.legend(bbox_to_anchor=(1.25, 1.2))
plt.show()

In [ ]:
import sklearn
CH_scores = []
resolutions = np.arange(0.2,2,0.1)

sc.tl.umap(fibroblastslognorm, random_state=2)
bk.bbknn(fibroblastslognorm,use_rep='X_pca', n_pcs=50)
for res in resolutions:
    sc.tl.leiden(fibroblastslognorm,resolution = res,n_iterations=-1, random_state=2)
    score = sklearn.metrics.calinski_harabasz_score(fibroblastslognorm.obsm['X_pca'],fibroblastslognorm.obs['leiden'])
    CH_scores.append(score)
    print(score)

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
plt.plot(resolutions, CH_scores)
plt.xlabel("Resolution")
plt.ylabel("CH score")
plt.rc('font', size=14)  
plt.rc('axes', labelsize=14) 

In [ ]:
import sklearn
resolutions = np.arange(0.2,2,0.1)
coefs = []
for res in resolutions:
    sc.tl.leiden(fibroblastslognorm,resolution = res,n_iterations=-1, random_state=2, key_added = f'leiden_{res}')
    coef = sklearn.metrics.silhouette_score(fibroblastslognorm.obsm['X_pca'],labels=fibroblastslognorm.obs[f'leiden_{res}'],metric='euclidean',sample_size=1000,random_state=2)
    coefs.append(coef)
print(coefs)

In [ ]:
plt.plot(resolutions, coefs)
plt.xlabel("Resolution")
plt.ylabel("Silhouette score")
plt.rc('font', size=14)  
plt.rc('axes', labelsize=14) 